In [2]:
# ! pip install booknlp_fr -U
from booknlp_fr import (
    load_tokenizer_and_embedding_model,
    get_embedding_tensor_from_tokens_df,
    load_text_file,
    load_tokens_df,
    save_tokens_df,
    load_entities_df,
)
from tqdm.auto import tqdm
import torch
import os
import pandas as pd

booknlp_fr package loaded successfully.


In [ ]:
detective_annotated_dataset = pd.read_csv('total_characters_annotato.csv')
tokens_entities_files_directory = "/BOOK_JEUNESSE"
embeddings_directory = "/attribute_embeddings"

detectives_features_dataset = []

for file_name, file_df in tqdm(detective_annotated_dataset.groupby("text_id")):
    file_name = os.path.splitext(file_name)[0]

    attributes_embeddings_path = os.path.join(embeddings_directory, f"{file_name}.attribute_embeddings")
    if not os.path.exists(attributes_embeddings_path):
        print(f"No attributes embeddings found for {file_name}")
        continue

    attribute_embeddings = torch.load(attributes_embeddings_path)

    tokens_df = load_tokens_df(file_name, tokens_entities_files_directory)
    
    attributes_tokens_df = tokens_df[tokens_df["is_PER_attribute"] == 1].copy().reset_index(drop=True) #ok

    entities_df = load_entities_df(file_name, tokens_entities_files_directory) # ok

    # fin qui tutto bene, la stampa dà qualcosa e non liste vuote
    
    for char_id, char_name, type, Gender in file_df[["char_id", "char_name", "type", "Gender"]].values:
        # char_id funziona, char_name funziona. Label dà NaN (normale)
        character_entities_df = entities_df[entities_df["COREF"] == char_id].copy()
        # il problema sembra essere character_entities_ df (il print(character_entities_df) dà liste vuote)

        character_head_ids = character_entities_df["head_id"].tolist()

        agent_attributes_ids = attributes_tokens_df[attributes_tokens_df["char_att_agent"].isin(character_head_ids)].index.tolist()
        patient_attributes_ids = attributes_tokens_df[attributes_tokens_df["char_att_patient"].isin(character_head_ids)].index.tolist()
        mod_attributes_ids = attributes_tokens_df[attributes_tokens_df["char_att_mod"].isin(character_head_ids)].index.tolist()
        pos_attributes_ids = attributes_tokens_df[attributes_tokens_df["char_att_poss"].isin(character_head_ids)].index.tolist()

        detectives_features_dataset.append(
            {"file_name": file_name,
             "char_name": char_name,
             "char_id": char_id,
             "type": type,
             "Gender": Gender,
             "agent_lemmas": attributes_tokens_df.loc[agent_attributes_ids, "lemma"].tolist(),
             "patient_lemmas": attributes_tokens_df.loc[patient_attributes_ids, "lemma"].tolist(),
             "mod_lemmas": attributes_tokens_df.loc[mod_attributes_ids, "lemma"].tolist(),
             "pos_lemmas": attributes_tokens_df.loc[pos_attributes_ids, "lemma"].tolist(),
             "agent_embeddings": attribute_embeddings[agent_attributes_ids],
             "patient_embeddings": attribute_embeddings[patient_attributes_ids],
             "mod_embeddings": attribute_embeddings[mod_attributes_ids],
             "pos_embeddings": attribute_embeddings[pos_attributes_ids],
             })
    # break

  0%|          | 0/123 [00:00<?, ?it/s]

No attributes embeddings found for 1867_Segur-comtesse-de_Le-Mauvais-Genie
No attributes embeddings found for 1877_Bruno-G_Le_Tour_de_la_France_par_deux_enfants
No attributes embeddings found for 1897_Margueritte-Victor_Poum_(aventures-d-un-petit-garçon)


In [ ]:
# ✅ Save it
detectives_features_dataset_path = "/book_jeunesse_features.pt"
torch.save(detectives_features_dataset, detectives_features_dataset_path)

# ✅ Load it back
loaded = torch.load(detectives_features_dataset_path)
print(loaded[0])

{'file_name': '1833_Girardin-Delphine-de_Contes-d-une-vieille-fille-a-ses-neveux', 'char_name': 'léon', 'char_id': 0, 'type': 1.0, 'Gender': 0.0, 'agent_lemmas': ['venir', 'obtenir', 'aimer', 'sauter', 'mettre', 'apercevoir', 'courir', 'mettre', 'préférer', 'aimer', 'reprendre', 'vouloir', 'vouloir', 'répondre', 'pouvoir', 'faire', 'retourner', 'regarder', 'revenir', 'regarder', 'éprouver', 'venir', 'interpréter', 'avoir', 'arriver', 'recommencer', 'vouloir', 'dire', 'reprendre', 'décider', 'désirer', 'préférer', 'avoir', 'prendre', 'préférer', 'prendre', 'répondre', 'reprendre', 'aimer', 'prendre', 'décider', 'faire', 'lever', 'devoir', 'lever', 'rappeler', 'obtenir', 'pressentir', 'paraître', 'suivre', 'avoir', 'entendre', 'trouver', 'voir', 'regarder', 'voir', 'contempler', 'obéir', 'secouer', 'faire', 'reculer', 'rapprocher', 'vouloir', 'remarquer', 'amuser', 'jouer', 'pouvoir', 'pouvoir', 'écrier', 'devoir', 'figurer', 'demander', 'faire', 'faire', 'porter', 'soupçonner', 'avoir',